## `pydp` - simple questions w/ and w/out privacy

**links:**
1. [github.com/OpenMined/PyDP](https://github.com/OpenMined/PyDP)
2. [pydp.readthedocs.io](https://pydp.readthedocs.io/en/latest/)

**code:** this notebooks follows the openmined's pydp titanic tutorial

**dataset:** `laveshjadon/ai-impact-on-students`

**what do we do:** each question we compute the answer twice:
once **without** differential privacy (the true statistic) and once **with** differential
privacy using pydp

**info pydp:** pydp (`python-dp`) is openmined's python wrapper using google's c++ differential-privacy
library. every private query in this notebook uses a laplace-based bounded aggregate (`BoundedMean`,
`Count`, `Min`, `Max`, `Median`, `BoundedStandardDeviation`) and takes a `privacy_budget` argument (epsilon).


In [1]:
# install the pydp package
!pip install python-dp --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 4.4 MB/s eta 0:00:00


In [2]:
import pydp as dp
from pydp.algorithms.laplacian import (
    BoundedSum,
    BoundedMean,
    BoundedStandardDeviation,
    Count,
    Max,
    Min,
    Median,
)
import pandas as pd
import statistics  # for computing stats without differential privacy

import kagglehub, os
path = kagglehub.dataset_download("laveshjadon/ai-impact-on-students")
files = [f for f in os.listdir(path) if f.endswith(".csv")]
df = pd.read_csv(os.path.join(path, files[0]))
df.head()

Using Colab cache for faster access to the 'ai-impact-on-students' dataset.


,Student_ID,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level
0,100001,Humanities,Senior,2.418,23.31,Copywriting/Drafting,Beginner,1,True,8.13,5,Allowed_With_Citation,6,2.393,86.44,High
1,100002,Medical,Junior,3.821,1.12,Ideation,Advanced,5,False,16.65,3,Allowed_With_Citation,9,3.696,69.39,Low
2,100003,Business,Freshman,3.398,21.26,Summarizing_Reading,Beginner,2,False,10.35,5,Strict_Ban,9,3.499,73.93,Medium
3,100004,Business,Senior,3.789,1.82,Copywriting/Drafting,Intermediate,4,False,15.23,2,Allowed_With_Citation,2,4.000,63.58,Medium
4,100005,STEM,Sophomore,3.635,9.29,Debugging/Troubleshooting,Advanced,4,False,12.55,4,Allowed_With_Citation,4,3.798,100.00,Medium


### Q1. what is the average weekly genai usage (hours) of students?
 `Weekly_GenAI_Hours` is our main continuous
variable (it ranges roughly 0-40).

In [3]:
# mean weekly genai hours WITHOUT differential privacy
def mean_hours() -> float:
    return statistics.mean(list(df["Weekly_GenAI_Hours"]))

In [4]:
# mean weekly genai hours WITH differential privacy
def private_mean(privacy_budget: float) -> float:
    x = BoundedMean(privacy_budget, lower_bound=0.0, upper_bound=40.0, dtype="float")
    return x.quick_result(list(df["Weekly_GenAI_Hours"]))

In [10]:
print("Mean (no DP):\t", mean_hours())
print("Mean (DP):\t", private_mean(privacy_budget=0.001)) # have to use really low values

Mean (no DP):	 8.4277522
Mean (DP):	 9.568048282860811


### Q2: how many students are in the dataset?
a simple count of rows

In [11]:
# total number of students WITHOUT differential privacy
def count_students() -> int:
    return df.count()["Weekly_GenAI_Hours"]

In [12]:
# total number of students WITH differential privacy
def private_count(privacy_budget: float) -> int:
    x = Count(privacy_budget, dtype="float")
    return x.quick_result(list(df["Weekly_GenAI_Hours"]))

In [20]:
print("Total (no DP):           " + str(count_students()))
print("Total (DP) eps = 0.1:    " + str(private_count(privacy_budget=0.1)))
print("Total (DP) eps = 0.5:    " + str(private_count(privacy_budget=0.5)))
print("Total (DP) eps = 0.01:   " + str(private_count(privacy_budget=0.01)))
print("Total (DP) eps = 0.0001: " + str(private_count(privacy_budget=0.001)))

Total (no DP):           50000
Total (DP) eps = 0.1:    50010
Total (DP) eps = 0.5:    49999
Total (DP) eps = 0.01:   49948
Total (DP) eps = 0.0001: 51236


### Q3: what is the lowest weekly genai usage?
the minimum of `Weekly_GenAI_Hours`.

In [21]:
# minimum weekly genai hours WITHOUT differential privacy
def min_hours() -> float:
    return df["Weekly_GenAI_Hours"].min()

In [22]:
# minimum weekly genai hours WITH differential privacy
def private_min(privacy_budget: float) -> float:
    # 0.0 and 40.0 are the search bounds for the value
    x = Min(privacy_budget, lower_bound=0.0, upper_bound=40.0, dtype="float")
    return x.quick_result(list(df["Weekly_GenAI_Hours"]))

In [26]:
print("Min (no DP):        " + str(min_hours()))
print("Min (DP) eps = 0.8  " + str(private_min(privacy_budget=0.8)))
print("Min (DP) eps = 0.1  " + str(private_min(privacy_budget=0.1)))
print("Min (DP) eps = 0.01 " + str(private_min(privacy_budget=0.01)))

Min (no DP):        0.0
Min (DP) eps = 0.8  0.010058221961111653
Min (DP) eps = 0.1  0.01097565303058241
Min (DP) eps = 0.01 0.11877615328528174


### Q4: what is the highest weekly genai usage?
the maximum of `Weekly_GenAI_Hours`

In [28]:
# maximum weekly genai hours WITHOUT differential privacy
def max_hours() -> float:
    return df["Weekly_GenAI_Hours"].max()

In [29]:
# maximum weekly genai hours WITH differential privacy
def private_max(privacy_budget: float) -> float:
    x = Max(privacy_budget, lower_bound=0.0, upper_bound=40.0, dtype="float")
    return x.quick_result(list(df["Weekly_GenAI_Hours"]))

In [35]:
print("Max (no DP):\t" + str(max_hours()))
print("Max (DP):\t" + str(private_max(privacy_budget=0.003)))

Max (no DP):	40.0
Max (DP):	35.74672485877442


### Q5: how many students use genai more than a certain number of hours?
counts students above a threshold

In [36]:
# count of students above "limit" hours WITHOUT differential privacy
def count_above(limit: float) -> int:
    return df[df.Weekly_GenAI_Hours > limit].count()["Weekly_GenAI_Hours"]

In [37]:
# count of students above "limit" hours WITH differential privacy
def private_count_above(privacy_budget: float, limit: float) -> int:
    x = Count(privacy_budget, dtype="float")
    return x.quick_result(list(df[df.Weekly_GenAI_Hours > limit]["Weekly_GenAI_Hours"]))

In [39]:
print("Above 30h (no DP):  " + str(count_above(30)))
print("Above 30h (DP):     " + str(private_count_above(privacy_budget=0.2, limit=30)))

Above 30h (no DP):  1604
Above 30h (DP):     1604


### Q6: how many students use genai less than a certain number of hours?
counts students below a threshold

In [40]:
# count of students below "limit" hours WITHOUT differential privacy
def count_below(limit: float) -> int:
    return df[df.Weekly_GenAI_Hours < limit].count()["Weekly_GenAI_Hours"]

In [41]:
# count of students below "limit" hours WITH differential privacy
def private_count_below(privacy_budget: float, limit: float) -> int:
    x = Count(privacy_budget, dtype="float")
    return x.quick_result(list(df[df.Weekly_GenAI_Hours < limit]["Weekly_GenAI_Hours"]))

In [45]:
print("Below 5h (no DP):            " + str(count_below(5)))
print("Below 5h (DP): eps = 0.8     " + str(private_count_below(privacy_budget=0.8, limit=5)))
print("Below 5h (DP): eps = 0.002   " + str(private_count_below(privacy_budget=0.002, limit=5)))

Below 5h (no DP):            22567
Below 5h (DP): eps = 0.8     22567
Below 5h (DP): eps = 0.002   22652


### Q7: a closer look at post-semester gpa
consider that you are analysing student outcomes and want to study `Post_Semester_GPA`
(which ranges 1-4).

In [48]:
# total students, and the paid vs free split
print("Total students:     ", len(df))
print("Paid subscription:  ", int(df["Paid_Subscription"].sum()))
print("No subscription:    ", int((~df["Paid_Subscription"]).sum()))

# subsets for the two groups
paid_df = df[df["Paid_Subscription"] == True]
free_df = df[df["Paid_Subscription"] == False]
print("paid group size:", len(paid_df), "| free group size:", len(free_df))

Total students:      50000
Paid subscription:   21154
No subscription:     28846
paid group size: 21154 | free group size: 28846


### Q8: what is the mean post-semester gpa?
mean of `Post_Semester_GPA`, with and without DP, for all students and for each subscription
group.

In [49]:
# mean gpa WITHOUT differential privacy
def mean_gpa(data) -> float:
    return statistics.mean(list(data["Post_Semester_GPA"]))

# mean gpa WITH differential privacy
def private_mean_gpa(data, privacy_budget: float) -> float:
    x = BoundedMean(privacy_budget, lower_bound=1.0, upper_bound=4.0, dtype="float")
    return x.quick_result(list(data["Post_Semester_GPA"]))

In [51]:
print("Mean GPA, all (no DP):   ", round(mean_gpa(df), 4))
print("Mean GPA, all (DP):      ", round(private_mean_gpa(df, 0.001), 4))
print()
print("Mean GPA, paid (no DP):  ", round(mean_gpa(paid_df), 4))
print("Mean GPA, paid (DP):     ", round(private_mean_gpa(paid_df, 0.001), 4))
print()
print("Mean GPA, free (no DP):  ", round(mean_gpa(free_df), 4))
print("Mean GPA, free (DP):     ", round(private_mean_gpa(free_df, 0.001), 4))

Mean GPA, all (no DP):    3.3493
Mean GPA, all (DP):       3.1643

Mean GPA, paid (no DP):   3.3525
Mean GPA, paid (DP):      3.3033

Mean GPA, free (no DP):   3.347
Mean GPA, free (DP):      3.2219


### Q9: what is the median post-semester gpa?
median of `Post_Semester_GPA`. pydp's `Median` uses the exponential mechanism internally to
pick a private median.

In [52]:
# median gpa WITHOUT differential privacy
def median_gpa(data) -> float:
    return statistics.median(list(data["Post_Semester_GPA"]))

# median gpa WITH differential privacy
def private_median_gpa(data, privacy_budget: float) -> float:
    x = Median(privacy_budget, lower_bound=1.0, upper_bound=4.0, dtype="float")
    return x.quick_result(list(data["Post_Semester_GPA"]))

In [53]:
print("Median GPA (no DP):   ", round(median_gpa(df), 4))
print("Median GPA (DP):      ", round(private_median_gpa(df, 0.001), 4))

Median GPA (no DP):    3.421
Median GPA (DP):       3.5695


### Q10: what is the standard deviation of post-semester gpa?
spread of `Post_Semester_GPA`, with and without DP.

In [55]:
# std of gpa WITHOUT differential privacy
def std_gpa(data) -> float:
    return statistics.stdev(list(data["Post_Semester_GPA"]))

# std of gpa WITH differential privacy
def private_std_gpa(data, privacy_budget: float) -> float:
    x = BoundedStandardDeviation(privacy_budget, lower_bound=1.0, upper_bound=4.0, dtype="float")
    return x.quick_result(list(data["Post_Semester_GPA"]))

In [56]:
print("Std GPA (no DP):   ", round(std_gpa(df), 4))
print("Std GPA (DP):      ", round(private_std_gpa(df, 0.001), 4))

Std GPA (no DP):    0.4957
Std GPA (DP):       0.9649
